# Gemma 4 E4B · llama.cpp Colab 대화
Google 공식 **Gemma 4 E4B QAT Q4_0 GGUF**를 llama.cpp로 실행하고 Gradio로 대화합니다.
공식 **Ubuntu x64 CUDA 12.8 사전 빌드 파일**을 사용하므로 소스 컴파일이 필요하지 않습니다.
모델 파일은 약 **5.15GB**이며, 기본 문맥은 4,096토큰입니다.

1. 기존 서버가 실행 중이라면 **런타임 → 연결 해제 및 런타임 삭제** 후 새 런타임을 만듭니다.
2. **런타임 → 런타임 유형 변경 → T4 GPU**를 선택합니다.
3. 셀을 위에서부터 실행합니다.
4. 대화창 셀을 실행하고 표시되는 Gradio 링크를 열어 대화합니다.

다운로드·압축 해제·설치·모델 로딩 로그를 표시합니다. 별도의 경과 시간 메시지는 출력하지 않습니다.
무료 Colab은 GPU와 사용 시간을 보장하지 않습니다. 노트북 UI를 벗어나 웹 UI를 주로 사용하는 방식은 제한되며 세션이 종료될 수 있습니다.
[Colab 정책](https://research.google.com/colaboratory/faq.html) · [모델](https://huggingface.co/google/gemma-4-E4B-it-qat-q4_0-gguf)


## 1. 실행 설정
처음에는 기본값을 사용하세요. 모델 파일은 Colab 임시 디스크에 저장됩니다. 세션이 초기화되면 다시 다운로드합니다.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

MODEL_REPO = "google/gemma-4-E4B-it-qat-q4_0-gguf"
MODEL_REVISION = "4b4a2c1d584be7264f87aac328a1bc739ce81b6c"
MODEL_FILENAME = "gemma-4-E4B_q4_0-it.gguf"
LLAMA_RELEASE = "b11433"
CUDA_VERSION = "12.8"
LLAMA_ARCHIVE = f"llama-{LLAMA_RELEASE}-bin-ubuntu-cuda-{CUDA_VERSION}-x64.tar.gz"
CUDA_ARCHIVE = f"cudart-{LLAMA_ARCHIVE}"
RELEASE_URL = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_RELEASE}"
UV_VERSION = "0.12.23"
PYTHON_PACKAGES = ["gradio==6.29.1", "huggingface-hub==2.1.1", "requests==2.34.2"]

WORK_DIR = Path("/content/gemma4_chat")
CONTEXT_SIZE = 4096
MAX_NEW_TOKENS = 512
BATCH_SIZE = 256
UBATCH_SIZE = 128
TEMPERATURE = 1.0
TOP_P = 0.95
TOP_K = 64
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000
GRADIO_PORT = 7860
SYSTEM_PROMPT = "사용자와 한국어로 대화하세요. 질문에 명확하고 자연스럽게 답하세요."
STARTUP_TIMEOUT = 900
REQUEST_TIMEOUT = 300

WORK_DIR.mkdir(parents=True, exist_ok=True)
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap,driver_version", "--format=csv,noheader"],
    check=True, capture_output=True, text=True,
)
print(gpu.stdout.strip(), flush=True)
SERVER_URL = f"http://{SERVER_HOST}:{SERVER_PORT}"
print(f"llama.cpp {LLAMA_RELEASE} · CUDA {CUDA_VERSION} · 문맥 {CONTEXT_SIZE}토큰", flush=True)


## 2. 사전 빌드 서버와 대화창 설치
Gradio 등 Python 패키지를 설치한 뒤 공식 llama.cpp CUDA 12.8 바이너리와 CUDA 라이브러리를 받습니다.
서버 파일은 약 172MB, CUDA 라이브러리는 약 594MB입니다. 다운로드 진행률과 압축 해제 파일을 표시합니다.
이 배포 파일은 Ubuntu 24.04용입니다. Colab의 최신 기본 런타임을 사용하세요.
CMake·CUDA 소스 빌드·vLLM 설치는 실행하지 않습니다.


In [ ]:
import threading
import time
import urllib.request

def run_logged(command, **kwargs):
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1, **kwargs,
    )

    def stream_output():
        with process.stdout:
            for line in process.stdout:
                print(line, end="", flush=True)

    reader = threading.Thread(target=stream_output, daemon=True)
    reader.start()
    try:
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        raise
    reader.join()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

uv_dir = WORK_DIR / "uv"
uv_binary = uv_dir / "uv"
if not uv_binary.exists():
    print("Python 패키지 설치 도구 uv 다운로드", flush=True)
    installer_url = f"https://github.com/astral-sh/uv/releases/download/{UV_VERSION}/uv-installer.sh"
    with urllib.request.urlopen(installer_url) as response:
        installer = response.read()
    installer_path = WORK_DIR / "uv-installer.sh"
    installer_path.write_bytes(installer)
    install_env = os.environ.copy()
    install_env.update(UV_INSTALL_DIR=str(uv_dir), UV_NO_MODIFY_PATH="1")
    run_logged(["sh", str(installer_path)], env=install_env)

print("Gradio와 모델 다운로드 도구 설치", flush=True)
run_logged([
    str(uv_binary), "pip", "install", "--system", "--python", sys.executable, *PYTHON_PACKAGES,
])

import requests
from tqdm.auto import tqdm

def download_archive(filename):
    destination = WORK_DIR / filename
    if destination.exists():
        print(f"기존 배포 파일 사용: {filename}", flush=True)
        return destination
    print(f"배포 파일 다운로드: {filename}", flush=True)
    partial_path = destination.with_suffix(destination.suffix + ".part")
    with requests.get(f"{RELEASE_URL}/{filename}", stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()
        total = int(response.headers.get("Content-Length", 0)) or None
        with partial_path.open("wb") as output, tqdm(
            total=total, unit="B", unit_scale=True, desc=filename,
            bar_format="{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt}",
        ) as progress:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)
                    progress.update(len(chunk))
    partial_path.replace(destination)
    return destination

binary_archive = download_archive(LLAMA_ARCHIVE)
cuda_archive = download_archive(CUDA_ARCHIVE)
release_dir = WORK_DIR / "llama-prebuilt"
release_dir.mkdir(parents=True, exist_ok=True)
print("llama.cpp 서버 압축 해제", flush=True)
run_logged(["tar", "-xzvf", str(binary_archive), "-C", str(release_dir)])
print("CUDA 라이브러리 압축 해제", flush=True)
run_logged(["tar", "-xzvf", str(cuda_archive), "-C", str(release_dir)])
SERVER_BINARY = release_dir / f"llama-{LLAMA_RELEASE}" / "llama-server"
CUDA_LIBRARY_DIR = release_dir / f"cudart-llama-{LLAMA_RELEASE}-bin-ubuntu-cuda-{CUDA_VERSION}-x64"
server_env = os.environ.copy()
server_env["LD_LIBRARY_PATH"] = os.pathsep.join(filter(None, [
    str(CUDA_LIBRARY_DIR), str(SERVER_BINARY.parent), server_env.get("LD_LIBRARY_PATH", ""),
]))
print("사전 빌드 서버 실행 확인", flush=True)
run_logged([str(SERVER_BINARY), "--version"], env=server_env)
print("서버와 대화창 설치 완료", flush=True)


## 3. GGUF 모델 다운로드
Google 공식 QAT Q4_0 GGUF 파일 하나를 다운로드합니다. 텍스트 대화에서는 이미지용 mmproj 파일이 필요하지 않습니다.
모델 크기는 약 5.15GB입니다. 다운로드 진행률을 표시하며, 같은 런타임에서 재실행하면 기존 파일을 재사용합니다.


In [ ]:
model_path = str(WORK_DIR / "models" / MODEL_FILENAME)
print(f"GGUF 모델 다운로드: {MODEL_REPO}/{MODEL_FILENAME}", flush=True)
download_script = """
import sys
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import enable_progress_bars
from tqdm import tqdm

class DownloadProgress(tqdm):
    def __init__(self, *args, **kwargs):
        kwargs["bar_format"] = "{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt}"
        super().__init__(*args, **kwargs)

enable_progress_bars()
hf_hub_download(
    repo_id=sys.argv[1], revision=sys.argv[2], filename=sys.argv[3],
    local_dir=sys.argv[4], tqdm_class=DownloadProgress,
)
print("GGUF 모델 다운로드 완료", flush=True)
"""
download_env = os.environ.copy()
download_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"
run_logged(
    [sys.executable, "-u", "-c", download_script, MODEL_REPO, MODEL_REVISION,
     MODEL_FILENAME, str(WORK_DIR / "models")],
    env=download_env,
)


## 4. llama.cpp 서버 실행
GGUF 모델을 GPU에 올리고 대화용 메모리를 할당합니다. 모델 로딩 및 GPU 메모리 로그를 실시간으로 표시합니다.
기본값은 문맥 4,096토큰, 동시 생성 1개이며, T4에서는 Flash Attention을 끕니다.
서버 준비가 완료되면 다음 대화창 셀을 실행하세요.


In [ ]:
import json
import time
import requests
import socket

if "server_process" in globals() and server_process.poll() is None:
    print("이전 모델 서버 종료", flush=True)
    server_process.terminate()
    server_process.wait(timeout=30)

with socket.socket() as port_check:
    try:
        port_check.bind((SERVER_HOST, SERVER_PORT))
    except OSError as error:
        raise RuntimeError(
            f"{SERVER_PORT} 포트가 이미 사용 중입니다. 설정 셀의 SERVER_PORT를 바꾸고 다시 실행하세요."
        ) from error
SERVER_URL = f"http://{SERVER_HOST}:{SERVER_PORT}"
print(f"llama.cpp 서버 시작: {SERVER_URL}", flush=True)

log_path = WORK_DIR / "llama-server.log"
server_command = [
    str(SERVER_BINARY), "--model", model_path, "--alias", MODEL_REPO,
    "--host", SERVER_HOST, "--port", str(SERVER_PORT),
    "--ctx-size", str(CONTEXT_SIZE), "--parallel", "1",
    "--n-gpu-layers", "99", "--batch-size", str(BATCH_SIZE),
    "--ubatch-size", str(UBATCH_SIZE), "--flash-attn", "off",
    "--jinja", "--chat-template-kwargs", json.dumps({"enable_thinking": False}),
    "--reasoning-format", "deepseek", "--reasoning-budget", "0",
]
with log_path.open("w") as log_file:
    server_process = subprocess.Popen(
        server_command, stdout=log_file, stderr=subprocess.STDOUT, env=server_env,
    )

print("GGUF 모델 로딩 및 GPU 메모리 할당", flush=True)
deadline = time.monotonic() + STARTUP_TIMEOUT
with log_path.open(errors="replace") as live_log:
    while time.monotonic() < deadline:
        new_logs = live_log.read()
        if new_logs:
            print(new_logs, end="", flush=True)
        if server_process.poll() is not None:
            print(live_log.read(), end="", flush=True)
            raise RuntimeError("모델 실행에 실패했습니다. 위 서버 로그를 확인하세요.")
        try:
            if requests.get(f"{SERVER_URL}/health", timeout=2).status_code == 200:
                print(live_log.read(), end="", flush=True)
                server_log_offset = live_log.tell()
                print("모델 준비 완료 — 아래 대화창 셀을 실행하세요.", flush=True)
                break
        except requests.RequestException:
            pass
        time.sleep(2)
    else:
        server_process.terminate()
        server_process.wait(timeout=30)
        print(live_log.read(), end="", flush=True)
        raise TimeoutError("모델 준비 시간이 초과되었습니다. 서버 로그를 확인하세요.")


## 5. Gradio 대화창
표시되는 `gradio.live` 링크를 열면 로그인 없이 바로 대화할 수 있습니다. 입력창에서 **Shift + Enter**로 전송합니다.

문맥 한도를 넘으면 대화를 지우고 새로 시작하세요.
이 셀은 대화 중에도 실행 상태를 유지하면서 서버 로그·요청 처리 단계·오류 상세를 표시합니다. 경과 시간은 출력하지 않습니다. 종료 셀을 실행하려면 먼저 이 셀의 중지 버튼을 누르세요.


In [ ]:
import gradio as gr
import traceback

chat_log_path = WORK_DIR / "gradio-chat.log"
chat_log_path.write_text("", encoding="utf-8")

def log_chat_action(text):
    with chat_log_path.open("a", encoding="utf-8") as chat_log:
        chat_log.write(f"[대화] {text}\n")

def follow_chat_logs():
    print("대화 중 서버 로그와 오류를 표시합니다. 로그 표시를 끝내려면 이 셀을 중지하세요.", flush=True)
    with log_path.open(errors="replace") as server_log, chat_log_path.open(encoding="utf-8") as chat_log:
        server_log.seek(globals().get("server_log_offset", 0))
        try:
            while True:
                for stream in (server_log, chat_log):
                    output = stream.read()
                    if output:
                        print(output, end="", flush=True)
                if server_process.poll() is not None:
                    for stream in (server_log, chat_log):
                        print(stream.read(), end="", flush=True)
                    raise RuntimeError("모델 서버가 종료되었습니다. 위 로그를 확인하세요.")
                time.sleep(0.2)
        except KeyboardInterrupt:
            print("로그 표시 중지. 대화창과 모델 서버를 닫으려면 종료 셀을 실행하세요.", flush=True)


def extract_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(
            block.get("text", "")
            for block in content
            if isinstance(block, dict) and block.get("type") == "text"
        )
    return ""

def generate_reply(message, history):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for entry in history:
        if entry["role"] in ("user", "assistant"):
            messages.append({
                "role": entry["role"],
                "content": extract_text(entry["content"]),
            })
    messages.append({"role": "user", "content": extract_text(message)})
    try:
        with requests.post(
            f"{SERVER_URL}/v1/chat/completions",
            json={
                "model": MODEL_REPO,
                "messages": messages,
                "max_tokens": MAX_NEW_TOKENS,
                "temperature": TEMPERATURE,
                "top_p": TOP_P,
                "top_k": TOP_K,
                "stream": True,
                "chat_template_kwargs": {"enable_thinking": False},
            },
            stream=True, timeout=(10, REQUEST_TIMEOUT),
        ) as response:
            log_chat_action(f"모델 서버 응답: HTTP {response.status_code}")
            if response.status_code >= 400:
                log_chat_action(f"모델 서버 오류 상세: {response.text}")
            if response.status_code == 400:
                raise gr.Error("요청을 처리할 수 없습니다. 문맥 한도를 넘었다면 대화를 지우고 다시 시작하세요.")
            response.raise_for_status()
            answer = ""
            # SSE를 바이트로 분리한 뒤 JSON에서 UTF-8로 디코딩합니다.
            for line in response.iter_lines(decode_unicode=False):
                if not line or not line.startswith(b"data: "):
                    continue
                data = line[6:]
                if data == b"[DONE]":
                    break
                event = json.loads(data)
                if event.get("error"):
                    log_chat_action(f"스트리밍 오류 상세: {event['error']}")
                    raise gr.Error("모델 실행 중 오류가 발생했습니다. 서버 로그를 확인하세요.")
                choices = event.get("choices", [])
                if choices:
                    text = choices[0].get("delta", {}).get("content") or ""
                    if text:
                        answer += text
                        yield answer
            if not answer:
                raise gr.Error("답변이 비어 있습니다. 서버 로그를 확인하세요.")
    except requests.RequestException as error:
        raise gr.Error("모델 서버 연결에 실패했습니다. 모델 실행 셀과 서버 로그를 확인하세요.") from error

def respond(message, history):
    log_chat_action("메시지 처리 시작")
    try:
        first_token = True
        for answer in generate_reply(message, history):
            if first_token:
                log_chat_action("답변 스트리밍 시작")
                first_token = False
            yield answer
        log_chat_action("답변 생성 완료")
    except Exception as error:
        log_chat_action("메시지 처리 실패\n" + traceback.format_exc())
        if isinstance(error, gr.Error):
            raise
        raise gr.Error(f"{type(error).__name__}: {error} — Colab 대화창 셀의 로그를 확인하세요.") from error

if "demo" in globals():
    demo.close()

demo = gr.ChatInterface(
    fn=respond,
    title="Gemma 4 E4B",
    description="한국어로 편하게 대화하세요.",
    textbox=gr.Textbox(placeholder="메시지를 입력하세요", lines=2),
    concurrency_limit=1,
)
demo.queue(max_size=1).launch(
    share=True,
    inline=True,
    server_name=SERVER_HOST,
    server_port=GRADIO_PORT,
    show_error=True,
    prevent_thread_lock=True,
)

follow_chat_logs()


## 6. 사용 종료
먼저 대화창 셀의 중지 버튼을 눌러 로그 표시를 끝낸 뒤 아래 종료 코드를 실행하세요.
대화를 마치면 이 셀로 공유 링크와 모델 서버를 닫습니다. 이후 **런타임 → 연결 해제 및 런타임 삭제**로 GPU 사용을 종료하세요.


In [ ]:
if "demo" in globals():
    demo.close()
if "server_process" in globals() and server_process.poll() is None:
    server_process.terminate()
    server_process.wait(timeout=30)
print("대화창과 모델 서버 종료 완료")
